# Legal Gemma 3n Fine-tuning with Unsloth

**Model**: Gemma 3n 2B/4B (multimodal: text + vision + audio)

**Layer Control**: Fine-tune language layers only (fits 15GB Colab), or enable vision/audio for more VRAM

**Datasets** (cached):
- Legal documents (Pile of Law, LexGLUE, MultiLexSum) - ~2-5 GB
- Codebase patterns (Svelte 5, evidence types, RAG logic) - ~30 MB

**Export**: TensorRT-LLM INT4 for Triton deployment (port 8099)

**Sources**:
- [Gemma 3n Multimodal](https://unsloth.ai/blog/gemma-3n)
- [Gemma 3n Documentation](https://docs.unsloth.ai/basics/gemma-3n-how-to-run-and-fine-tune)
- [Unsloth Vision Models](https://huggingface.co/collections/unsloth/vision-multimodal-models)

## 1. Setup & Installation

In [ ]:
# Install Unsloth (latest from GitHub)
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install additional dependencies
!pip install bitsandbytes accelerate peft trl transformers datasets huggingface_hub pillow

In [ ]:
import torch
from unsloth import FastVisionModel, is_bfloat16_supported
from transformers import TrainingArguments, TextStreamer
from trl import SFTTrainer
from datasets import load_dataset, concatenate_datasets, Dataset
import json
import re
from pathlib import Path

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    
    # Recommend model based on VRAM
    if vram_gb >= 40:
        print("\nRecommended: gemma-3n-4b-bnb-4bit (fits 40GB A100)")
    elif vram_gb >= 15:
        print("\nRecommended: gemma-3n-2b-bnb-4bit or gemma-3n-4b-bnb-4bit (fits 15GB T4)")
    else:
        print("\nRecommended: gemma-3n-2b-bnb-4bit (fits <15GB)")
        print("Warning: Fine-tune language layers ONLY (vision/audio needs >15GB)")

## 2. Model Configuration

**Gemma 3n models** (multimodal: text + vision + audio):
- `unsloth/gemma-3n-2b-bnb-4bit` - 2B params, ~4-6 GB VRAM, 32K context
- `unsloth/gemma-3n-4b-bnb-4bit` - 4B params, ~6-8 GB VRAM, 32K context

**Layer fine-tuning control**:
- Language layers only: ~6-8 GB VRAM (fits free Colab/Kaggle)
- Vision + audio layers: >15 GB VRAM (needs paid Colab A100)

In [ ]:
# Model configuration
MODEL_NAME = "unsloth/gemma-3n-2b-bnb-4bit"  # Change to gemma-3n-4b-bnb-4bit if >15GB VRAM
MAX_SEQ_LENGTH = 2048  # Gemma 3n supports up to 32K with Unsloth

# LoRA configuration
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Layer fine-tuning control (adjust based on VRAM)
FINETUNE_VISION_LAYERS = False      # Set True if VRAM > 15GB and you want vision fine-tuning
FINETUNE_LANGUAGE_LAYERS = True     # Always True for text fine-tuning
FINETUNE_ATTENTION_MODULES = True   # True for attention layers
FINETUNE_MLP_MODULES = True         # True for MLP layers

print(f"Model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"\nFine-tuning config:")
print(f"  Vision layers: {FINETUNE_VISION_LAYERS}")
print(f"  Language layers: {FINETUNE_LANGUAGE_LAYERS}")
print(f"  Attention modules: {FINETUNE_ATTENTION_MODULES}")
print(f"  MLP modules: {FINETUNE_MLP_MODULES}")

## 3. Load & Cache Legal Datasets

**Cached datasets** (~2-5 GB total):
1. Pile of Law (legal documents)
2. LexGLUE (contract provisions, case law)
3. MultiLexSum (legal summarization)
4. FineTome (instruction-following)
5. GSM8K (reasoning)

In [ ]:
# Helper: standardize text columns
def standardize_text(example):
    """Convert 'text' column to string"""
    if 'text' not in example:
        return example
    
    if isinstance(example['text'], list):
        # Handle list of dicts with 'value' key (FineTome conversations)
        example['text'] = ' '.join([
            item['value'] if isinstance(item, dict) and 'value' in item else str(item)
            for item in example['text']
        ])
    elif not isinstance(example['text'], str):
        example['text'] = str(example['text'])
    
    return example

print("Loading legal datasets (this will cache to ~/.cache/huggingface)...\n")

# 1. FineTome (instruction-following)
print("[1/7] FineTome...")
dataset1 = load_dataset("mlabonne/FineTome-100k", split="train[:10000]")
dataset1 = dataset1.rename_column('conversations', 'text')
print(f"  ✓ {len(dataset1)} examples")

# 2. GSM8K (math reasoning)
print("[2/7] GSM8K...")
dataset2 = load_dataset("openai/gsm8k", "main", split="train[:5000]")
dataset2 = dataset2.rename_column('question', 'text')
print(f"  ✓ {len(dataset2)} examples")

# 3. Pile of Law (legal documents)
print("[3/7] Pile of Law...")
pile_of_law = load_dataset("lamblamb/pile_of_law_subset", split="train[:20000]")
print(f"  ✓ {len(pile_of_law)} examples")

# 4. LexGLUE LEDGAR (contract provisions)
print("[4/7] LexGLUE LEDGAR...")
ledgar = load_dataset("lex_glue", "ledgar", split="train[:10000]")
print(f"  ✓ {len(ledgar)} examples")

# 5. MultiLexSum (legal summarization)
print("[5/7] MultiLexSum...")
multilexsum = load_dataset("allenai/multi_lexsum", name="v20230518", split="train[:5000]")
multilexsum = multilexsum.rename_column('summary/short', 'text')
print(f"  ✓ {len(multilexsum)} examples")

# 6. LexGLUE Case Hold (legal reasoning)
print("[6/7] LexGLUE Case Hold...")
lexglue_case_hold = load_dataset("lighteval/lexglue", name="case_hold", split="train[:5000]")
lexglue_case_hold = lexglue_case_hold.rename_column('input', 'text')
print(f"  ✓ {len(lexglue_case_hold)} examples")

# 7. LexGLUE SCOTUS (Supreme Court)
print("[7/7] LexGLUE SCOTUS...")
lexglue_scotus = load_dataset("lighteval/lexglue", name="scotus", split="train[:5000]")
lexglue_scotus = lexglue_scotus.rename_column('input', 'text')
print(f"  ✓ {len(lexglue_scotus)} examples")

# Standardize all datasets
print("\nStandardizing text columns...")
datasets_to_combine = []
for ds in [dataset1, dataset2, pile_of_law, ledgar, multilexsum, lexglue_case_hold, lexglue_scotus]:
    ds = ds.select_columns(['text'])
    ds = ds.map(standardize_text, num_proc=4)
    datasets_to_combine.append(ds)

# Combine
legal_dataset = concatenate_datasets(datasets_to_combine)
print(f"\n✅ Total legal examples: {len(legal_dataset):,}")

## 4. Add Codebase Patterns

Domain-specific patterns from your legal AI codebase

In [ ]:
# Codebase domain knowledge (evidence pipeline + Svelte 5 + RAG)
codebase_patterns = [
    {"text": "Evidence type auto-detection: application/pdf → documentary, image/* → photo, video/* → video, audio/* → audio. Legal keyword reclassification: 'sworn statement' → testimonial, 'DNA analysis' → scientific, 'expert opinion' → expert, 'witness testimony' → witness_statement."},
    {"text": "Svelte 5 runes API: $state() creates reactive state, $derived() creates computed values, $derived.by() for complex derivations with blocks, $effect() for side effects, $props() for component properties. NO Svelte 4 patterns: no 'export let', no '$:', no 'on:click'."},
    {"text": "Legal entity extraction patterns: EMAIL (RFC 5322 format), PHONE (E.164 international format), CITATION (Bluebook legal citation format), STATUTE (U.S. Code format like '42 U.S.C. § 1983'), DATE (ISO 8601), MONEY (currency symbols with amounts)."},
    {"text": "Forensic pattern detection: SSN matches /\\d{3}-\\d{2}-\\d{4}/, credit card matches /\\d{4}[- ]?\\d{4}[- ]?\\d{4}[- ]?\\d{4}/, contact density threshold is 5+ contacts per 1000 characters, legal keyword density threshold is 10+ keywords per 1000 words."},
    {"text": "ACE (Adaptive Context Engine) context assembly: Fetches 7 parallel data sources (case information, evidence metadata with forensic flags, relevant statutes, legal precedents, timeline events, relationship graph, user practice area context). Token budget allocation: caseContext 300 tokens, evidenceMetadata 200 tokens, ragChunks 400 tokens, statutes 200 tokens, precedents 300 tokens, timeline 200 tokens, relationships 200 tokens, userContext 100 tokens. Total: 1900 tokens."},
    {"text": "RAG evidence upload pipeline (8 stages): 1) MinIO cloud storage upload with SHA-256 hash, 2) Text extraction (pdf-parse for PDFs, Tesseract OCR for images, native for text files), 3) Legal-aware structure chunking (respects ARTICLE/SECTION/§ boundaries), 4) 768-dimension embeddings via embeddinggemma model, 5) Dual storage in pgvector + Qdrant vector databases, 6) Entity extraction (EMAIL, PHONE, CITATION, STATUTE, DATE, MONEY, PERSON, ORG), 7) Forensic pattern detection (SSN, credit cards, contact density, legal keywords), 8) LLM summarization via gemma3-legal model (non-fatal, graceful degradation)."},
    {"text": "Evidence type enum (16 legal categories): document (generic), photo (image evidence), video (recorded footage), audio (recordings), documentary (contracts/agreements), testimonial (sworn statements), demonstrative (exhibits), witness_statement (witness accounts), forensic (lab results), scientific (technical analysis), expert (expert opinions), physical (tangible objects), digital (electronic records), real (original items), circumstantial (indirect evidence), hearsay (out-of-court statements)."},
    {"text": "bits-ui v2 Dialog component pattern (Svelte 5): <Dialog.Root bind:open={isOpen}> contains trigger and portal. <Dialog.Portal> wraps overlay and content. <Dialog.Overlay /> provides backdrop. <Dialog.Content> contains title, description, close button. For transitions use forceMount prop with child snippet: <Dialog.Overlay forceMount>{#snippet child({ props, open })}{#if open}<div {...props} transition:fade>overlay</div>{/if}{/snippet}</Dialog.Overlay>. NEVER use transition props directly."},
    {"text": "Qdrant vector search dual-vector strategy: Content embedding (768-dim from document text via embeddinggemma) gets 60% weight, signature embedding (768-dim from file path + imports + exports via code structure) gets 40% weight. Hybrid search combines: 1) Fuse.js fuzzy recall (fast, finds candidates), 2) Qdrant dual-vector rerank (precise, scores with 0.6 content + 0.4 signature). Returns top-k chunks with scores."},
    {"text": "TensorRT-LLM deployment: Gemma 3n model converts to INT4 quantized TRT engine via trtllm-build. Triton Inference Server runs on port 8099 with model repository layout: models/gemma3n_legal/1/model.plan (TRT engine), config.pbtxt (max_batch_size 8, instance_group GPU). Client calls via HTTP POST to /v2/models/gemma3n_legal/infer with JSON payload. GPU arbiter manages VRAM mutex between Ollama (port 11434) and TRT-LLM (port 8099) using Redis locks."}
]

# Convert to Dataset
codebase_dataset = Dataset.from_list(codebase_patterns)
print(f"Codebase patterns: {len(codebase_dataset)} examples")

# Combine with legal datasets
combined_dataset = concatenate_datasets([legal_dataset, codebase_dataset])
print(f"Total training examples: {len(combined_dataset):,}")
print(f"\nDataset size: ~{len(combined_dataset) * 0.5 / 1024:.1f} MB (text-only)")

## 5. Format for Gemma 3n Chat Template

In [ ]:
def format_for_gemma3n_chat(example):
    """Convert text to Gemma 3n conversation format"""
    text = example.get('text', '')
    
    # Determine instruction based on content
    if any(kw in text.lower() for kw in ['evidence', 'forensic', 'rag', 'upload']):
        instruction = "Explain this legal evidence processing concept:"
    elif any(kw in text.lower() for kw in ['$state', '$derived', '$effect', 'svelte', 'runes']):
        instruction = "Explain this Svelte 5 programming pattern:"
    elif any(kw in text.lower() for kw in ['statute', 'citation', 'u.s.c', 'bluebook']):
        instruction = "Explain this legal citation or statute:"
    elif any(kw in text.lower() for kw in ['tensorrt', 'triton', 'trt-llm', 'gpu']):
        instruction = "Explain this AI inference deployment concept:"
    else:
        instruction = "Explain the following legal concept:"
    
    # Gemma 3n chat format
    return {
        "conversations": [
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": text}
        ]
    }

# Apply formatting
print("Formatting dataset for Gemma 3n chat template...")
train_dataset = combined_dataset.map(format_for_gemma3n_chat, remove_columns=['text'], num_proc=4)
print(f"✅ Formatted {len(train_dataset):,} examples")

# Preview
print("\nExample conversation:")
print(json.dumps(train_dataset[0]['conversations'], indent=2))

## 6. Load Gemma 3n Model

In [ ]:
print(f"Loading {MODEL_NAME}...\n")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect (bfloat16 if supported)
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Supports bfloat16: {is_bfloat16_supported()}")

## 7. Add LoRA Adapters with Layer Control

**Layer fine-tuning options**:
- `finetune_language_layers=True` (always) - Text understanding
- `finetune_vision_layers=False` (default) - Needs >15GB VRAM
- `finetune_attention_modules=True` - Attention layers
- `finetune_mlp_modules=True` - MLP layers

In [ ]:
print("Adding LoRA adapters...\n")

model = FastVisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    # Layer-specific fine-tuning (adjust based on VRAM)
    finetune_vision_layers=FINETUNE_VISION_LAYERS,         # False = language-only (fits 15GB)
    finetune_language_layers=FINETUNE_LANGUAGE_LAYERS,     # True = fine-tune text layers
    finetune_attention_modules=FINETUNE_ATTENTION_MODULES, # True = fine-tune attention
    finetune_mlp_modules=FINETUNE_MLP_MODULES,             # True = fine-tune MLP
    use_gradient_checkpointing="unsloth",  # Unsloth's 30% faster checkpointing
    random_state=42,
)

print("LoRA adapter configuration:")
print(f"  r (rank): {LORA_R}")
print(f"  alpha: {LORA_ALPHA}")
print(f"  dropout: {LORA_DROPOUT}")
print(f"\nLayer fine-tuning:")
print(f"  Vision layers: {FINETUNE_VISION_LAYERS}")
print(f"  Language layers: {FINETUNE_LANGUAGE_LAYERS}")
print(f"  Attention modules: {FINETUNE_ATTENTION_MODULES}")
print(f"  MLP modules: {FINETUNE_MLP_MODULES}")
print()

# Print trainable parameters
model.print_trainable_parameters()

## 8. Configure Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./gemma3n-legal-outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduce to 1 if OOM
    gradient_accumulation_steps=8,  # Effective batch size = 16
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    warmup_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    max_grad_norm=1.0,
    dataloader_num_workers=4,  # Parallel data loading
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Precision: {'bfloat16' if is_bfloat16_supported() else 'float16'}")
print(f"  Optimizer: {training_args.optim}")

## 9. Initialize Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    dataset_text_field="conversations",
    packing=False,  # Disable for multimodal models
)

print("✅ Trainer initialized")

## 10. Train

In [ ]:
print("\n" + "="*70)
print("Starting training...")
print("="*70 + "\n")

trainer_stats = trainer.train()

print("\n" + "="*70)
print("Training complete!")
print("="*70)
print(f"\nTraining time: {trainer_stats.metrics['train_runtime']:.2f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Samples/second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"Total steps: {trainer_stats.metrics.get('train_steps', 'N/A')}")

## 11. Test Inference

In [ ]:
# Enable inference mode
FastVisionModel.for_inference(model)

# Test prompts
test_prompts = [
    "Explain how evidence type detection works in a legal AI system.",
    "What are Svelte 5 runes and how do they differ from Svelte 4?",
    "Describe the RAG evidence upload pipeline."
]

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

for prompt in test_prompts:
    print("\n" + "="*70)
    print(f"Prompt: {prompt}")
    print("="*70)
    
    inputs = tokenizer(
        [{"role": "user", "content": prompt}],
        return_tensors="pt",
        padding=True
    ).to("cuda")
    
    outputs = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1
    )
    print("\n")

## 12. Save LoRA Adapters

In [ ]:
# Save LoRA adapters (small, ~100-500 MB)
model.save_pretrained("gemma3n-legal-lora")
tokenizer.save_pretrained("gemma3n-legal-lora")

print("✅ LoRA adapters saved to: gemma3n-legal-lora/")
print("Size: ~100-500 MB (adapters only)")

## 13. Export Merged Model (16-bit or 4-bit)

In [ ]:
# Merge LoRA weights into base model

# Option 1: 16-bit merged model (~5 GB for Gemma 3n 2B)
print("Exporting 16-bit merged model...")
model.save_pretrained_merged(
    "gemma3n-legal-merged-16bit",
    tokenizer,
    save_method="merged_16bit"
)
print("✅ 16-bit model saved to: gemma3n-legal-merged-16bit/ (~5 GB)")

# Option 2: 4-bit merged model (~1.5 GB for Gemma 3n 2B)
print("\nExporting 4-bit merged model...")
model.save_pretrained_merged(
    "gemma3n-legal-merged-4bit",
    tokenizer,
    save_method="merged_4bit"
)
print("✅ 4-bit model saved to: gemma3n-legal-merged-4bit/ (~1.5 GB)")

## 14. Export for TensorRT-LLM + Triton Deployment

**Convert to TRT-LLM INT4 engine** (run on your local machine):

```bash
# 1. Download merged model from Colab
# Use files.download('gemma3n-legal-merged-4bit.zip') or manual download

# 2. Convert to TensorRT-LLM checkpoint
python TensorRT-LLM/examples/gemma/convert_checkpoint.py \
  --model_dir gemma3n-legal-merged-16bit \
  --output_dir trt_checkpoints/gemma3n-legal \
  --dtype float16 \
  --tp_size 1  # Tensor parallelism (1 GPU)

# 3. Build TensorRT engine (INT4 quantization)
trtllm-build \
  --checkpoint_dir trt_checkpoints/gemma3n-legal \
  --output_dir trt_engines/gemma3n-legal \
  --gemm_plugin float16 \
  --use_weight_only \
  --weight_only_precision int4 \
  --max_batch_size 8 \
  --max_input_len 2048 \
  --max_output_len 512 \
  --max_beam_width 1

# 4. Deploy via Triton Inference Server
# Model repository structure:
models/
└── gemma3n_legal/
    ├── config.pbtxt
    └── 1/
        └── model.plan  # Copy from trt_engines/gemma3n-legal/

# 5. Start Triton (port 8099)
docker run -d --gpus all --rm \
  -p 8099:8000 \
  -v $(pwd)/models:/models \
  nvcr.io/nvidia/tritonserver:24.01-py3 \
  tritonserver --model-repository=/models

# 6. Update src/lib/server/trt-llm.ts
# Change TENSORRT_URL to http://localhost:8099
# Model name: gemma3n_legal
```

**Triton config.pbtxt**:
```protobuf
name: "gemma3n_legal"
backend: "tensorrtllm"
max_batch_size: 8

model_transaction_policy {
  decoupled: True
}

instance_group [
  {
    count: 1
    kind: KIND_GPU
  }
]
```

In [ ]:
# Zip merged model for download
!zip -r gemma3n-legal-merged-4bit.zip gemma3n-legal-merged-4bit/

print("\n✅ Model packaged: gemma3n-legal-merged-4bit.zip")
print("\nDownload to your local machine, then:")
print("1. Convert to TensorRT-LLM checkpoint")
print("2. Build INT4 TRT engine")
print("3. Deploy via Triton on port 8099")
print("4. Update src/lib/server/trt-llm.ts")
print("\nSee cell above for full commands.")

# Optional: Auto-download in Colab
# from google.colab import files
# files.download('gemma3n-legal-merged-4bit.zip')

## 15. Upload to Hugging Face (Optional)

In [ ]:
# Login to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

# Push to HF Hub
model.push_to_hub_merged(
    "your-username/gemma3n-legal-2b",  # Change to your HF username
    tokenizer,
    save_method="merged_16bit",
    token=""  # Uses login token
)

print("✅ Model uploaded!")
print("Access at: https://huggingface.co/your-username/gemma3n-legal-2b")

---

## Summary

**What we trained**:
- Base: Gemma 3n 2B (multimodal: text + vision + audio)
- Data: 60K+ legal documents + 10 codebase patterns (~2-5 GB cached)
- Method: QLoRA 4-bit with Unsloth (1.5x faster, 50% less VRAM)
- Layers: Language + attention + MLP (vision disabled for 15GB VRAM)
- Output: 4-bit merged model (~1.5 GB)

**Performance**:
- Training time: ~1-2 hours on T4 GPU (15GB VRAM)
- VRAM usage: ~6-8 GB during training
- Inference: ~4-6 GB VRAM with INT4 TRT-LLM

**Deployment**:
1. Download gemma3n-legal-merged-4bit.zip
2. Convert to TensorRT-LLM INT4 engine (~1.2 GB)
3. Deploy via Triton Inference Server (port 8099)
4. Wire into `/api/vision/analyze` endpoint
5. GPU arbiter manages VRAM between Ollama (11434) + TRT-LLM (8099)

**Sources**:
- [Gemma 3n Multimodal Blog](https://unsloth.ai/blog/gemma-3n)
- [Gemma 3n Documentation](https://docs.unsloth.ai/basics/gemma-3n-how-to-run-and-fine-tune)
- [Unsloth Vision Models](https://huggingface.co/collections/unsloth/vision-multimodal-models)
- [TensorRT-LLM Gemma Example](https://github.com/NVIDIA/TensorRT-LLM/tree/main/examples/gemma)